# 02 — Feature build + labels

Aggregate 1-minute klines → 20-minute decision bars and produce the one-sided excursion label `y_k = 1[ ln(H_{k+1} / C_k) ≥ α ]` (D2 contract).

**One function does everything:** `wagie.training.compute_labels`.

Streaming features (the ~600 default) are computed inside the engine at run time — see `wagie.features.catalog.default_streaming_features`. We don't pre-compute them: the sealed Pipeline is the single source of truth, and prod ≡ backtest depends on bit-identical batch≡stream behaviour.

In [ ]:
from pathlib import Path
from wagie.training import compute_labels

raw = Path("data/synthetic/btcusdt_1m.parquet")
assert raw.is_file(), f"run 01_data_download first; missing {raw}"

df = compute_labels(raw, m_minutes=20)
print(f"decision bars: {len(df)}")
print("columns:", df.columns)
df.head(5)

Label statistics — at the default α (90% quantile of train log-excursions), positives should be ~10% of the train set, fewer once you cross into test territory:

In [ ]:
import polars as pl
summary = (
    df.filter(pl.col("label").is_not_null())
      .select([
          pl.col("label").mean().alias("positive_rate"),
          pl.col("label").sum().alias("n_positive"),
          pl.col("log_excursion").mean().alias("mean_excursion"),
          pl.col("log_excursion").quantile(0.9).alias("q90_excursion"),
      ])
)
summary

Save the labelled bars for the offline trainer:

In [ ]:
out = Path("data/cleansed_data/btcusdt_20m_labeled.parquet")
out.parent.mkdir(parents=True, exist_ok=True)
df.write_parquet(out)
print(f"wrote {out}")

## Optional — warm the streaming Mondrian-ACI calibrator from the train slice

If you have a real offline classifier this becomes very useful (set `q_init_by_regime` on the calibrator). With a uniform-p baseline it is a no-op, included here to show the API.

In [ ]:
from wagie.training import warm_online_quantiles

q_init = warm_online_quantiles(
    raw, m_minutes=20, alphas=(0.05, 0.10, 0.20),
    train_frac=0.6, n_regimes=3,
)
for a, per_regime in q_init.items():
    print(f"alpha={a:.2f}  q_init_by_regime={per_regime}")

Move on to **03_offline_train** to fit a CatBoost on `data/cleansed_data/...`.